In [48]:
import pandas as pd
import numpy as np
import requests                  # for calling APIs and downloading websites
import yaml                      # for loading credentials
from bs4 import BeautifulSoup    # for parsing html
import time                      # to put the system to sleep
import random                    # for random numbers
import re                        # regular expressions

pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings("ignore")

In [3]:
# Required to be able to access a wikipedia page
headers = {"User-Agent": "PPOL5203 class project"}

# step 1: find a website with information you want to collect
url = "https://en.wikipedia.org/wiki/ChatGPT"

# step 2: understand the website

# step 3: write code to collect one realization of the data
page = requests.get(url, headers=headers)
soup = BeautifulSoup(page.content, 'html.parser')

# grab release and strip it 
release = soup.select('th.infobox-label:-soup-contains("Release") + td')
cleaned = [i.get_text() for i in release]
cleaned[0].strip()

text = "\n".join([i.strip() for i in cleaned])
print(text[0:1000])


November 30, 2022(3 years ago) (2022-11-30)[1]


In [4]:
# helper function 
def get_infobox_value(soup, label):
    """helper function that collects the value associated with the label

    Args:
        soup: parsed HTML of a Wikipedia page.
        label: infobox label to look for: "Sector", "Release"
    Returns:
        str: text of the matching value cell, stripped of extra spaces
        None: if the label is not on the page
    """
    label = soup.select(f'th.infobox-label:-soup-contains("{label}") + td')

    if len(label) > 0:
        return label[0].get_text().strip()
    return None

In [6]:
def scrape_wiki_tool(url=None):
    """Collect release year, and type from a universities about Wikipedia page.
    
    Downloads the page, reads the infobox with get_infobox_value(), and pulls the
    four-digit year out of the release text with a regular expression.
    
    Args:
        url (str): full Wikipedia URL for one university page.
    Returns:
        dict: one row of university facts facts with keys
    """
    # get access to the website
    page = requests.get(url, headers = headers)
    soup = BeautifulSoup(page.content, 'html.parser')
    if page.status_code == 200:   
        uni_type = get_infobox_value(soup, "Type")
        established = get_infobox_value(soup, "Established")

        # clean the type into just "Public" or "Private"
        if uni_type and "private" in uni_type.lower():
            sector = "Private"
        elif uni_type and "public" in uni_type.lower():
            sector = "Public"
        else:
            sector = None
        
        year = re.search(r'\d{4}', str(established))
        
        return {"url": url,
                "type_raw": uni_type,
                "sector": sector,
                "established_year": int(year.group()) if year else None}

scrape_wiki_tool("https://en.wikipedia.org/wiki/Chittagong_University_of_Engineering_%26_Technology")

{'url': 'https://en.wikipedia.org/wiki/Chittagong_University_of_Engineering_%26_Technology',
 'type_raw': 'Public',
 'sector': 'Public',
 'established_year': 1968}

In [44]:
tool_urls = {
    "Daffodil International University": "https://en.wikipedia.org/wiki/Daffodil_International_University",
    "American International University-Bangladesh": "https://en.wikipedia.org/wiki/American_International_University-Bangladesh",
    "University of Dhaka": "https://en.wikipedia.org/wiki/University_of_Dhaka",
    "Jahangirnagar University": "https://en.wikipedia.org/wiki/Jahangirnagar_University",
    "Chittagong University of Engineering and Technology": "https://en.wikipedia.org/wiki/Chittagong_University_of_Engineering_%26_Technology",
}

# loop through and collect
results = []
for tool, url in tool_urls.items():
    #scrape the content
    row = scrape_wiki_tool(url)
    if row is not None:
        row["university"] = tool
        results.append(row)
    # put the system to sleep for a random draw of time
    time.sleep(random.uniform(2, 5))   

In [45]:
# step 5: save
tools_df = pd.DataFrame(results)
tools_df.to_csv("wiki_ai_tools.csv", index=False)
tools_df

,url,type_raw,sector,established_year,university
0,https://en.wikipedia.org/wiki/Daffodil_Interna...,Private research university,Private,2002,Daffodil International University
1,https://en.wikipedia.org/wiki/American_Interna...,Private research university,Private,1994,American International University-Bangladesh
2,https://en.wikipedia.org/wiki/University_of_Dhaka,Public research university,Public,1921,University of Dhaka
3,https://en.wikipedia.org/wiki/Jahangirnagar_Un...,"Public, research university",Public,1970,Jahangirnagar University
4,https://en.wikipedia.org/wiki/Chittagong_Unive...,Public,Public,1968,Chittagong University of Engineering and Techn...


In [53]:
path = "Data/Career_Anxiety_AI_Bangladesh.xlsx" 

students = pd.read_excel(path)
students

#get the right data first with university column
#students_copy = students.merge(tools_df[["university", "sector", "established_year"]],
#                          on="university", how="left")

,participant_id,age,gender,academic_year,ai_knowledge,ai_tools_used,career_path,ai_tool_perception,ai_future_perspective,ai_replace_jobs,ai_takeover_time,career_anxiety,consent
0,77BE5Q,24,Female,4th Year,Medium,"ChatGPT, Quiltbot, Grammarly",Researcher,I don't know for now,"I believe AI will mostly assist humans, with l...",No,Never,Low,Yes
1,12UT7F,22,Female,2nd Year,Medium,"ChatGPT, Google Gemini, Photomath",Researcher,Google Gemini,I think AI will replace some human jobs but al...,Partially,11–20 years,Medium,Yes
2,25AD1U,22,Female,2nd Year,Low,"ChatGPT, Google Gemini, Microsoft Copilot",Project Manager,Microsoft Copilot,I think AI will replace some human jobs but al...,No,Never,No Anxiety,Yes
3,85ZY9V,21,Female,2nd Year,Low,"ChatGPT, Google Gemini, Don’t know the name",Nutritionist,NaN,I believe AI will significantly replace human ...,Partially,21–50 years,No Anxiety,Yes
4,18HV7K,21,Male,2nd Year,Medium,"ChatGPT, Google Gemini, Microsoft Copilot",Researcher,ChatGPT,I strongly believe AI cannot take over human a...,No,Never,Low,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3151,28ZG8G,21,Male,3rd Year,Medium,"ChatGPT, Google Gemini, Microsoft Copilot",Electrical Engineer,ChatGPT,I think AI will replace some human jobs but al...,Partially,50+ years,Low,Yes
3152,57MJ7H,21,Male,1st Year,Medium,"ChatGPT, Google Gemini, Deepseek",SQA Engineer,ChatGPT,I believe AI will significantly replace human ...,Partially,1–5 years,Medium,Yes
3153,33MZ0S,21,Male,1st Year,High,"ChatGPT, Deepseek, Perplexity",Robotics Engineer,ChatGPT,I think AI will replace some human jobs but al...,Partially,6–10 years,Low,Yes
3154,74UD5I,20,Female,4th Year,Medium,"ChatGPT, Microsoft Copilot, Perplexity",Corporate Job Holder,NaN,I strongly believe AI will completely take ove...,Partially,21–50 years,Medium,Yes
